Nhận diện vũ khí

In [1]:
# Khai báo thư viện
import os
import random
import shutil
import torch
import time
from google.colab import drive

# Khai báo phương thức
def init(drive_path):
  # Test kết nối GPU của Server
  !nvidia-smi

  # Mount drive lên Server
  if not os.path.exists("/content/drive"):
      drive.mount('/content/drive')

  # Tạo folder dự án
  if not os.path.exists(drive_path):
    os.makedirs(drive_path, exist_ok=True)
  %cd $drive_path

  # Clone YOLOv5
  if not os.path.exists('yolov5'):
    !git clone https://github.com/ultralytics/yolov5
  %cd yolov5
  %pip install -qr requirements.txt

  # Tải yolov5m.pt
  if not os.path.exists('yolov5m.pt'):
    torch.hub.download_url_to_file('https://github.com/ultralytics/yolov5/releases/download/v7.0/yolov5m.pt', 'yolov5m.pt')

def count_png_files(directory):
  total_png = 0
  for root, dirs, files in os.walk(directory):
      png_in_current_folder = [f for f in files if f.lower().endswith('.png')]
      total_png += len(png_in_current_folder)
  return total_png

def safe_move_all(src, dst):
  if os.path.exists(src) and os.listdir(src):
      os.makedirs(dst, exist_ok=True)
      for f in os.listdir(src):
          shutil.move(os.path.join(src, f), os.path.join(dst, f))

def epoch_hien_tai(checkpoint_dir):
  res = 0
  if os.path.exists(checkpoint_dir):
      ckpt = torch.load(checkpoint_dir, map_location='cpu', weights_only=False)
      res = ckpt['epoch'] + 1
  return res

def sinh_mini_dataset(drive_path, dts_name, mini_dataset_exists, learned_epochs, target_epoches, mini_img_train, mini_lab_train,
                      mini_img_val, mini_lab_val, orig_img_train, orig_lab_train, orig_img_val, orig_lab_val):
  if learned_epochs < target_epoches:
      root_dir = f"{drive_path}/Safe-Sight-CV/{dts_name}-mini"
      mini_dataset_exists = os.path.exists(root_dir) and count_png_files(root_dir) > 0

      if not mini_dataset_exists:
          print("Đang chuẩn bị Mini Dataset để tối ưu tốc độ...")
          os.makedirs(mini_img_train, exist_ok=True)
          os.makedirs(mini_lab_train, exist_ok=True)
          os.makedirs(mini_img_val, exist_ok=True)
          os.makedirs(mini_lab_val, exist_ok=True)

          all_train_imgs = [f for f in os.listdir(orig_img_train) if f.lower().endswith('.jpg')]
          if all_train_imgs:
              selected = random.sample(all_train_imgs, min(1000, len(all_train_imgs)))
              for img in selected:
                  lab = img.rsplit('.', 1)[0] + '.txt'
                  if os.path.exists(os.path.join(orig_lab_train, lab)):
                      shutil.move(os.path.join(orig_img_train, img), os.path.join(mini_img_train, img))
                      shutil.move(os.path.join(orig_lab_train, lab), os.path.join(mini_lab_train, lab))

          for f in os.listdir(orig_img_val):
              shutil.move(os.path.join(orig_img_val, f), os.path.join(mini_img_val, f))
          for f in os.listdir(orig_lab_val):
              shutil.move(os.path.join(orig_lab_val, f), os.path.join(mini_lab_val, f))

          yaml_content = f"train: {mini_img_train}\nval: {mini_img_val}\nnc: 2\nnames: ['Knife', 'Handgun']"
          with open(f"{YOLO_PATH}/data/{YAML_FILE}", 'w') as f:
              f.write(yaml_content)

          print("Chờ Drive đồng bộ 45s trước khi Train...")
          time.sleep(45)
          return True
  return False

def huan_luyen(model_dir, learned_epochs, checkpoint_dir, target_epochs, yaml_file, drive_path, exp):
  %cd {model_dir}
  weights = checkpoint_dir if os.path.exists(checkpoint_dir) else "yolov5m.pt"
  !python train.py --img 320 --batch 16 --epochs {target_epochs} \
      --data data/{yaml_file} --weights {weights} \
      --project {drive_path}/runs --name {exp} --exist-ok --cache ram --patience 0

def move_to_original(mini_dataset_exists, learned_epochs, target_epochs, mini_img_train, orig_img_train, mini_lab_train, orig_lab_train,
                     mini_img_val, orig_img_val, mini_lab_val, orig_lab_val, drive_path, dts_name, yolo_path, yaml_file):
  if mini_dataset_exists and learned_epochs >= target_epochs:
    if os.path.exists(mini_img_train) and len(os.listdir(mini_img_train)) > 0:
      print("\nĐang trả dữ liệu về vị trí gốc...")

      safe_move_all(mini_img_train, orig_img_train)
      safe_move_all(mini_lab_train, orig_lab_train)
      safe_move_all(mini_img_val, orig_img_val)
      safe_move_all(mini_lab_val, orig_lab_val)

      print("Đang đợi Google Drive đồng bộ (180 giây)...")
      time.sleep(180)

      if len(os.listdir(mini_img_train)) == 0:
          shutil.rmtree(f"{drive_path}/Safe-Sight-CV/{dts_name}-mini")
          yaml_content = f"train: {orig_img_train}\nval: {orig2_img_val}\nnc: 2\nnames: ['Knife', 'Handgun']"
          with open(f"{YOLO_PATH}/data/{YAML_FILE}", 'w') as f:
              f.write(yaml_content)
          print("Đã dọn dẹp thư mục tạm.")

def kiem_nghiem(drive_path, exp, yaml_file):
  !python val.py --weights {drive_path}/runs/{exp}/weights/best.pt \
                --data data/{yaml_file} \
                --img 320 --half

# Khai báo đường dẫn folder
DRIVE_PATH   = "/content/drive/MyDrive/Noi_Dung_Nhay_Cam"
YOLO_PATH    = f"{DRIVE_PATH}/yolov5"
EXP_NAME     = "weapon_model"
DTS_NAME     = "weapon-dataset"
YAML_FILE    = "Weapon_detection.yaml"
MINI_DATASET_EXISTS = False
CHECKPOINT   = f"{DRIVE_PATH}/runs/{EXP_NAME}/weights/last.pt"
ORIG_IMG_TRAIN = f"{DRIVE_PATH}/Safe-Sight-CV/{DTS_NAME}/images/train"
ORIG_LAB_TRAIN = f"{DRIVE_PATH}/Safe-Sight-CV/{DTS_NAME}/labels/train"
ORIG_IMG_VAL   = f"{DRIVE_PATH}/Safe-Sight-CV/{DTS_NAME}/images/val"
ORIG_LAB_VAL   = f"{DRIVE_PATH}/Safe-Sight-CV/{DTS_NAME}/labels/val"
MINI_IMG_TRAIN = f"{DRIVE_PATH}/Safe-Sight-CV/{DTS_NAME}-mini/images/train"
MINI_LAB_TRAIN = f"{DRIVE_PATH}/Safe-Sight-CV/{DTS_NAME}-mini/labels/train"
MINI_IMG_VAL   = f"{DRIVE_PATH}/Safe-Sight-CV/{DTS_NAME}-mini/images/val"
MINI_LAB_VAL   = f"{DRIVE_PATH}/Safe-Sight-CV/{DTS_NAME}-mini/labels/val"

init(DRIVE_PATH)

# Nhập số EPOCH muốn train
try:
    val = input("Nhập TỔNG SỐ EPOCH mục tiêu (VD: 50, 100): ")
    TARGET_EPOCHS = int(val)
except:
    TARGET_EPOCHS = 50

# Sinh bộ dataset_mini ngẫu nhiên (hoàn thành đủ số epoch mới sinh bộ mới)
LEARNED_EPOCHS = epoch_hien_tai(CHECKPOINT)

print(f"--- Hiện tại: {LEARNED_EPOCHS} Epochs | Mục tiêu: {TARGET_EPOCHS} ---")

MINI_DATASET_EXISTS = sinh_mini_dataset(DRIVE_PATH, DTS_NAME, MINI_DATASET_EXISTS, LEARNED_EPOCHS, TARGET_EPOCHS, MINI_IMG_TRAIN, MINI_LAB_TRAIN,
                  MINI_IMG_VAL, MINI_LAB_VAL, ORIG_IMG_TRAIN, ORIG_LAB_TRAIN, ORIG_IMG_VAL, ORIG_LAB_VAL)

# Bắt đầu training
huan_luyen(YOLO_PATH, LEARNED_EPOCHS, CHECKPOINT, TARGET_EPOCHS, YAML_FILE, DRIVE_PATH, EXP_NAME)

LEARNED_EPOCHS = epoch_hien_tai(CHECKPOINT)

# Chuyển dữ liệu training về vị trí gốc
move_to_original(MINI_DATASET_EXISTS, LEARNED_EPOCHS, TARGET_EPOCHS, MINI_IMG_TRAIN, ORIG_IMG_TRAIN, MINI_LAB_TRAIN, ORIG_LAB_TRAIN,
                 MINI_IMG_VAL, ORIG_IMG_VAL, MINI_LAB_VAL, ORIG_LAB_VAL, DRIVE_PATH, DTS_NAME, YOLO_PATH, YAML_FILE)

print("QUÁ TRÌNH TRAIN ĐÃ HOÀN THÀNH!")

# Chạy thử mô hình đã train lên tập val để xem kết quả
kiem_nghiem(DRIVE_PATH, EXP_NAME, YAML_FILE)

Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
      36/49      2.12G    0.02744    0.01034   0.000988         42        320:  95% 179/188 [00:35<00:01,  5.13it/s]/content/drive/.shortcut-targets-by-id/1beMkBMZ2Wb5qOsfGBbJIu67oxhmkxuCh/Noi_Dung_Nhay_Cam/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
      36/49      2.12G    0.02747    0.01034  0.0009886         35        320:  96% 180/188 [00:36<00:01,  4.68it/s]/content/drive/.shortcut-targets-by-id/1beMkBMZ2Wb5qOsfGBbJIu67oxhmkxuCh/Noi_Dung_Nhay_Cam/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
      36/49      2.12G    0.02744    0.01034  0.0009843         39        320:  96% 181/188 [00:36<00:01,  4.71it/s]/content/drive/.shortcut-targets-by-id/1beMkBMZ2Wb5